# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', 'Dataset')}: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect the record sets, and for each record set, list its `@id`, available fields and columns, all by their `@id` as required.

In [ ]:
# List all record sets and their details
print("Available record sets (referenced by @id):")
if hasattr(dataset.metadata, 'record_sets'):
    for record_set in dataset.metadata.record_sets:
        print(f"- RecordSet @id: {record_set.id}")
        print(f"  Name: {getattr(record_set, 'name', '')}")
        if hasattr(record_set, 'fields'):
            print("  Fields (by @id):")
            for field in record_set.fields:
                print(f"    - {field.id} (name: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')})")
        if hasattr(record_set, 'columns'):
            print("  Columns (by @id):")
            for column in record_set.columns:
                print(f"    - {column.id} (name: {getattr(column, 'name', '')}, source: {getattr(column, 'source', '')})")
        print()
else:
    # Sometimes the record sets may be missing from metadata
    print("No record sets defined in the schema.\nTo investigate, try dataset.metadata.to_json() or refer to dataset.records() directly.")

# For quick access, we will enumerate all available record_set @id's for use in next steps
record_set_ids = []
if hasattr(dataset.metadata, 'record_sets'):
    record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("All record set @id's detected:")
print(record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We extract all records from each of the detected record sets, and load the first available record set into a DataFrame for further processing.

In [ ]:
# Extract data from each record set
dataframes = {}
if record_set_ids:
    for record_set in record_set_ids:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
    main_id = record_set_ids[0]
    print(f"Loaded columns for the first record set: {main_id}")
    print(dataframes[main_id].columns.tolist())
    dataframes[main_id].head()
else:
    # If no record_set explicitly, use the default (single-table) approach
    print("No record sets found. Trying default records() access...")
    records = list(dataset.records())
    df = pd.DataFrame(records)
    print(df.columns.tolist())
    df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes. All references use canonical `@id` field names.

> **Note**: Replace `<numeric_field_id>` or `<group_field_id>` with the actual field or column `@id` you want to explore, as identified in Section 2 for your dataset.

In [ ]:
# Example: Select a numeric field for analysis (by @id, adjust to match your field names)

if record_set_ids:
    df = dataframes[main_id].copy()

    # ATTENTION: Pick a real numeric field @id from data overview above, for example 'PHQ9_score' or similar
    # By default, we'll try to auto-detect first numeric column (int/float)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical field (e.g. 'gender', or similar)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (showing mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found to group by.")
    else:
        print("No numeric field detected in this record set.")
else:
    print("No record set data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For typical survey datasets, it's common to plot histograms for assessment scores (e.g., GAD-7, PHQ-9), pie charts for demographic splits, and boxplots for numerical fields by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If group_field_id was found in previous cell
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we've loaded a FAIR²-structured mental health survey dataset following the Croissant schema, explored its content (by record set, field, and column `@id`), performed basic data cleaning and feature engineering, and visualized key numeric variables. 

Further analysis and domain-specific visualizations can be built using the identified fields (by `@id`) and leveraging their meaning as documented in the dataset metadata. For deeper research, always refer to the full metadata and schema at the source URL.